# Setup

In [ ]:
# Run ONLY once. Working directory should be 02-machine-translation
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", Path.cwd())

Working directory: /Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/31-projects-2026/02-machine-translation


In [2]:
# Load autoreload extension for Jupyter Notebook
%load_ext autoreload
%autoreload 2

# Import Config, Dataset, and other necessary modules
from src.machine_translation.config import TatoebaConfig
from src.machine_translation.dataset import TatoebaData

# Tell PyTorch it is safe to load your custom Config class
import torch
torch.serialization.add_safe_globals([TatoebaConfig, TatoebaData])

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

# Set Pytorch Lightning logging level to WARNING to reduce verbosity
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
logging.getLogger("lightning_fabric").setLevel(logging.WARNING)

In [3]:
# Set up a logger for "tatoeba" level to DEBUG for this notebook
logger = logging.getLogger("tatoeba")
logger.setLevel(logging.DEBUG)

In [ ]:
# Set up a logger for "tatoeba" level to INFO for this notebook
logger = logging.getLogger("tatoeba")
logger.setLevel(logging.INFO)

# 01 `TatoebaData` class

## 01 Loading the dataset

In [7]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config)

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from datasets/tatoeba...
DEBUG:tatoeba.dataset:  Setting up Tatoeba dataset splits...
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157839, validation=39460, test=24514


In [6]:
# Inspect datasets or fetch a batch from a dataloader
print("Train samples:", len(data.train_data))
print("Sample:", data.train_data[0])

Train samples: 157839
Sample: {'source_text': 'Tom tried to break up the fight.', 'target_text': 'Tom trató de disolver la pelea.', 'source_lang': 'eng', 'target_lang': 'spa'}


In [ ]:
# Inspect a batch from DataLoader BEFORE tokenization
train_loader = data.train_dataloader()
batch = next(iter(train_loader))
print("Batch type:", type(batch), "Batch keys:", batch.keys())
print("Batch size:", len(batch['source_text']), len(batch['target_text']))
print("Sample source text:", batch['source_text'][0])

Batch type: <class 'dict'> Batch keys: dict_keys(['source_text', 'target_text', 'source_lang', 'target_lang'])
Batch size: 32 32
Sample source text: He can't be ill.


## 02 Training a BPE tokenizer from scratch

In [28]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config)

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from datasets/tatoeba...
DEBUG:tatoeba.dataset:  Training BPE tokenizer with vocab_size=10000 and max_seq_length=256...


DEBUG:tatoeba.dataset:  BPE tokenizer trained: vocab_size=10000, pad_id=0
DEBUG:tatoeba.dataset:  Setting up Tatoeba dataset splits...


DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157839, validation=39460, test=24514


In [29]:
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")
print(f"<pad> ID: {tokenizer.token_to_id('<pad>')}")
print(f"<unk> ID: {tokenizer.token_to_id('<unk>')}")
print(f"<s> ID: {tokenizer.token_to_id('<s>')}")
print(f"</s> ID: {tokenizer.token_to_id('</s>')}")

Vocabulary size: 10000
<pad> ID: 0
<unk> ID: 1
<s> ID: 2
</s> ID: 3


In [30]:
# Inspect one English/Spanish pair and its encoded result
example = data.train_data[0]

source_text = example["source_text"]
target_text = example["target_text"]

source_encoding = tokenizer.encode(source_text)
target_encoding = tokenizer.encode(target_text)

print("English:", source_text)
print("English tokens:", source_encoding.tokens)
print("English IDs:", source_encoding.ids)
print()
print("Spanish:", target_text)
print("Spanish tokens:", target_encoding.tokens)
print("Spanish IDs:", target_encoding.ids)

English: Tom tried to break up the fight.
English tokens: ['Tom', 'tried', 'to', 'break', 'up', 'the', 'fight', '.']
English IDs: [217, 1826, 196, 1600, 444, 214, 3006, 17]

Spanish: Tom trató de disolver la pelea.
Spanish tokens: ['Tom', 'trató', 'de', 'dis', 'olver', 'la', 'pelea', '.']
Spanish IDs: [217, 4654, 201, 462, 3069, 210, 7759, 17]


In [31]:
# Confirm padding and truncation work
encodings = tokenizer.encode_batch([
    data.train_data[0]["source_text"],
    data.train_data[1]["source_text"],
])

for encoding in encodings:
    print("length:", len(encoding.ids))
    print("ids:", encoding.ids)
    print("attention mask:", encoding.attention_mask)

length: 8
ids: [217, 1826, 196, 1600, 444, 214, 3006, 17]
attention mask: [1, 1, 1, 1, 1, 1, 1, 1]
length: 8
ids: [4522, 43, 315, 9384, 17, 0, 0, 0]
attention mask: [1, 1, 1, 1, 1, 0, 0, 0]


In [32]:
type(encodings), len(encodings), type(encodings[0])

(list, 2, tokenizers.Encoding)

In [33]:
encodings[0]

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

## 03 Data Limit for Quick Experiments

In [4]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=None)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Local cache not found at datasets/tatoeba. Downloading Tatoeba dataset from Hugging Face...


Generating validation split:   0%|          | 0/197299 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24514 [00:00<?, ? examples/s]

DEBUG:tatoeba.dataset:  Training BPE tokenizer with max_vocab_size=10000 and max_seq_length=256...


DEBUG:tatoeba.dataset:  BPE tokenizer trained: vocab_size=10000, pad_id=0
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  Setting up Tatoeba dataset splits...
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba


DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157,839, validation=39,460, test=24,514
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer.json.
DEBUG:tatoeba.dataset:  Vocab size: 10,000


In [10]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=None)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 157,839
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=157,839, validation=39,460, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer.json.
DEBUG:tatoeba.dataset:  Vocab size: 10,000


In [11]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  === _train_tokenizer() call ===
DEBUG:tatoeba.dataset:  Training BPE tokenizer with max_vocab_size=10,000 and max_seq_length=256...
DEBUG:tatoeba.dataset:  BPE tokenizer trained: vocab_size=5,652, pad_id=0
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba


DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


In [12]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


## 04 Custom `collate_fn` for the DataLoader

In [13]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Create a TatoebaData object using the config
data = TatoebaData(config, data_limit=1000)  # Limit to 1000 samples for quick experiments

# Manually trigger download/caching and setup splits for notebook exploration
data.prepare_data()
data.setup()

DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded Tatoeba dataset: train=800, validation=200, test=24,514
DEBUG:tatoeba.dataset:  === _load_tokenizer_from_disk() call ===
DEBUG:tatoeba.dataset:  Loaded tokenizer from datasets/tokenizers/tokenizer_limit_1000.json.
DEBUG:tatoeba.dataset:  Vocab size: 5,652


In [19]:
data.train_data[0]

{'source_text': 'None of us are opposed to his ideas.',
 'target_text': 'Ninguno de nosotros está en contra de sus ideas.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [23]:
# Create train dataloader for the training dataset
train_dataloader = data.train_dataloader()

# Fetch a batch from the dataloader and inspect it
batch = next(iter(train_dataloader))

DEBUG:tatoeba.dataset:  === _collate_fn() call ===
DEBUG:tatoeba.dataset:  Batch type: <class 'list'>
DEBUG:tatoeba.dataset:  Batch keys: ['source_text', 'target_text', 'source_lang', 'target_lang']
DEBUG:tatoeba.dataset:  Batch example source_text: We should have gotten married.
DEBUG:tatoeba.dataset:  Batch example target_text: Deberíamos habernos casado.
DEBUG:tatoeba.dataset:  Batch source_texts length: 32
DEBUG:tatoeba.dataset:  Batch example source_text: We should have gotten married.
DEBUG:tatoeba.dataset:  Batch target_texts length: 32
DEBUG:tatoeba.dataset:  Batch example target_text: <s> Deberíamos habernos casado. </s>
DEBUG:tatoeba.dataset:  Source encodings length: 32
DEBUG:tatoeba.dataset:  Source encoding example encodings: Encoding(num_tokens=17, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
DEBUG:tatoeba.dataset:  Source encoding example ids: [286, 495, 217, 4302, 2174, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
DEBUG:tatoeba.